# ENSP355 – AI in Cyber Security Lab 1
## Recognizing Cyber Threats Using Simple AI Rules

**Platform:** Google Colab / Python  
**Aim:** Classify messages as Safe, Phishing, Malware or Ransomware using transparent keyword rules, calculate accuracy, assign risk levels and document responsible use.

**Before running:** Upload the faculty-provided `simple_cyber_messages.csv` when prompted. The CSV must contain these columns: `message_id`, `message`, and `correct_category`. Use only sanitized classroom messages. Do not open links or attachments from suspicious messages.

## General Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from google.colab import files
from IPython.display import display

print("Please upload simple_cyber_messages.csv")
uploaded = files.upload()

# Read the uploaded CSV file
csv_name = next((name for name in uploaded if name.lower().endswith(".csv")), None)
if csv_name is None:
    raise ValueError("Please upload the faculty-provided CSV file.")

df = pd.read_csv(csv_name)

required_columns = {"message_id", "message", "correct_category"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"CSV is missing required columns: {sorted(missing_columns)}")

df["message"] = df["message"].fillna("").astype(str)
df["correct_category"] = df["correct_category"].astype(str).str.strip()

print("Dataset shape:", df.shape)
display(df.head())

## Q1. Manual Threat Identification (5 Marks)

**Aim:** Review the first ten messages and identify words that influenced the threat decision.

The keyword matches below are shown as a review aid. Read the message and confirm the important words manually. A message should be called Safe only when no clear threat indicator is present and its context is legitimate.

In [ ]:
# Important indicators based on the lab manual's threat categories.
Q1_KEYWORDS = {
    "Phishing": [
        "click here", "verify", "password", "urgent", "otp",
        "prize", "bank", "login", "blocked", "suspended"
    ],
    "Malware": [
        "malware", "virus", "infected", "attachment",
        "trojan", "spyware", "executable"
    ],
    "Ransomware": [
        "encrypted", "ransom", "bitcoin", "restore files",
        "locked files", "unlock", "payment demand"
    ]
}

def find_important_words(message):
    message_lower = str(message).lower()
    matches = []
    for words in Q1_KEYWORDS.values():
        matches.extend(word for word in words if word in message_lower)
    return ", ".join(dict.fromkeys(matches)) if matches else "No clear threat keyword; check legitimate context"

manual_table = df.head(10)[
    ["message_id", "message", "correct_category"]
].copy()

manual_table["important_words"] = manual_table["message"].apply(find_important_words)
display(manual_table)

print("Observation: Safe messages should still be checked in context because a message may contain ordinary words that appear suspicious, or an attacker may use unfamiliar wording.")

## Q2. Keyword Lists and Rule-Based Classifier (10 Marks)

**Aim:** Build a transparent classifier. It counts matching keywords in each category. If every score is zero, it returns Safe; otherwise it returns the category with the highest score.

In [ ]:
RULE_KEYWORDS = {
    "Phishing": [
        "click here", "verify", "password", "urgent",
        "otp", "prize", "bank", "login", "blocked"
    ],
    "Malware": [
        "malware", "virus", "infected", "attachment",
        "trojan", "spyware", "executable"
    ],
    "Ransomware": [
        "encrypted", "ransom", "bitcoin", "restore files",
        "locked files", "unlock", "payment demand"
    ]
}

def classify_message(text):
    text = str(text).lower()
    scores = {
        category: sum(word in text for word in words)
        for category, words in RULE_KEYWORDS.items()
    }

    # max() follows the dictionary order if two categories tie.
    predicted = max(scores, key=scores.get)

    if scores[predicted] == 0:
        return "Safe", "", 0

    matched = [
        word for word in RULE_KEYWORDS[predicted]
        if word in text
    ]
    return predicted, ", ".join(matched), scores[predicted]

# Test at least four messages, including one Safe example.
test_messages = [
    "The department meeting is scheduled for tomorrow.",
    "Urgent: click here to verify your password.",
    "The attachment contains a virus and a trojan.",
    "Your files are encrypted. Pay bitcoin to restore files."
]

for sample in test_messages:
    print(sample)
    print("Prediction:", classify_message(sample))
    print("-" * 70)

**Observation / limitation:** The rules are easy to understand because every decision is based on visible keywords. However, keyword rules may miss synonyms, spelling variations, context and new attack language.

## Q3. Complete Dataset Classification and Accuracy (10 Marks)

**Aim:** Apply the classifier to every message, compare predictions with the faculty-provided correct labels and calculate accuracy.

Accuracy = (Number of correct predictions / Total predictions) × 100.

In [ ]:
# Apply the classifier to every message.
results = df["message"].apply(classify_message)

df[["predicted_category", "matched_keywords", "keyword_score"]] = pd.DataFrame(
    results.tolist(), index=df.index
)

df["correct"] = df["correct_category"] == df["predicted_category"]

correct_predictions = int(df["correct"].sum())
incorrect_predictions = int((~df["correct"]).sum())
accuracy = accuracy_score(df["correct_category"], df["predicted_category"])

output_columns = [
    "message_id", "correct_category", "predicted_category",
    "matched_keywords", "keyword_score", "correct"
]
display(df[output_columns])

print("Correct predictions:", correct_predictions)
print("Incorrect predictions:", incorrect_predictions)
print(f"Accuracy: {accuracy * 100:.2f}%")

**Observation:** Accuracy measures the proportion of messages classified correctly in this dataset. Classroom accuracy can be optimistic because the dataset is small and its examples may use the same keywords as the rules. Real messages can contain new wording, mixed signals or misleading context. A false positive is a safe message labelled as a threat. A false negative is a threat labelled as Safe.

## Q4. Risk Level, Recommended Action and Bar Chart (10 Marks)

**Aim:** Assign a risk level and cautious action based on the predicted category and keyword score.

In [ ]:
def add_risk_and_action(row):
    if row["predicted_category"] == "Safe":
        return "Low", "Continue normal caution."

    if row["predicted_category"] == "Ransomware" or row["keyword_score"] >= 2:
        return "High", "Do not click/open; isolate if required and report."

    return "Medium", "Do not interact; verify the sender and report."

risk_output = df.apply(add_risk_and_action, axis=1)
df[["risk_level", "recommended_action"]] = pd.DataFrame(
    risk_output.tolist(), index=df.index
)

display(df[[
    "message_id", "predicted_category", "keyword_score",
    "risk_level", "recommended_action"
]])

category_order = ["Safe", "Phishing", "Malware", "Ransomware"]
category_counts = df["predicted_category"].value_counts().reindex(
    category_order, fill_value=0
)

category_counts.plot(kind="bar", edgecolor="black")
plt.title("Predicted Message Categories")
plt.xlabel("Category")
plt.ylabel("Number of Messages")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print("High-risk results require verification by a human analyst because keyword rules can make mistakes and the recommended action may affect real systems.")

### Live demonstration – enter a new message

Try a harmless example first, then try a clearly suspicious example. Do not paste real passwords, personal information or live suspicious links.

In [ ]:
new_message = input("Enter a sanitized sample message: ")
category, matched, score = classify_message(new_message)

demo_row = pd.DataFrame([{
    "predicted_category": category,
    "keyword_score": score
}])
risk, action = add_risk_and_action(demo_row.iloc[0])

print("Predicted category:", category)
print("Matched keywords:", matched if matched else "None")
print("Keyword score:", score)
print("Risk level:", risk)
print("Recommended action:", action)

## Q5. Responsible Use, Reflection and File Saving (5 Marks)

**Responsible-use points**
1. **Privacy:** Remove personal identifiers such as names, email addresses, account IDs and phone numbers before analysis.
2. **Bias:** Keyword lists may work better for some writing styles or languages than others.
3. **False alarms:** Safe messages may contain warning words, causing false positives; missed attacks can also cause harm.
4. **Accountability:** The person using the system remains responsible for the final decision.
5. **Human review:** High-risk or unclear cases must be checked by a human analyst.

**Conclusion:** Rule-based AI provides a simple and transparent way to identify possible cyber threats. It can classify messages using keywords and assign a basic risk level with a recommended action. Its decisions are easy to explain because the rules are visible. However, it may fail when messages use unfamiliar words, different languages or misleading context. It can also produce false positives and false negatives. Therefore, the system should support—not replace—a human cyber-security analyst. Privacy must be protected and serious or unclear cases must be reviewed by a person.

In [ ]:
# Save the completed result table.
output_file = "Lab1_Final_Result.csv"
df.to_csv(output_file, index=False)
print("Saved:", output_file)

# Download the CSV to your computer from Google Colab.
files.download(output_file)

## Final Submission Checklist

- [ ] Run every notebook cell from top to bottom.
- [ ] Upload the faculty-provided `simple_cyber_messages.csv`.
- [ ] Review the ten-message manual identification table.
- [ ] Record the prediction table, correct/incorrect counts and accuracy.
- [ ] Record the risk/action table and bar chart.
- [ ] Run the live new-message demonstration.
- [ ] Keep the responsible-use notes and conclusion.
- [ ] Download `Lab1_Final_Result.csv`.
- [ ] Share the Colab notebook link with the instructor.

**Important:** The exact accuracy and result tables depend on the faculty-provided CSV. The notebook does not invent a replacement dataset.